# MMELON Model Fine-Tuning

Fine-tune the MMELON (Multi-Modal Encoder for Learning on Molecules) model on hit prediction data.

## Overview

This notebook demonstrates how to fine-tune MMELON for binary classification tasks in drug discovery:

**What is MMELON?**
- Multi-modal molecular encoder combining multiple molecular representations
- Integrates graph neural networks, fingerprints, and other molecular features
- Pre-trained on large-scale molecular datasets

**Workflow:**
1. **Setup & Configuration** - Environment setup and hyperparameter configuration
2. **Data Preparation** - Load screening data and create data loaders
3. **Model Initialization** - Load pre-trained MMELON and configure prediction head
4. **Training Setup** - Configure PyTorch Lightning trainer
5. **Fine-Tuning** - Train the model on your dataset
6. **Evaluation** - Assess model performance and save results

**Use Cases:**
- Hit prediction from screening campaigns
- Activity classification (active/inactive)
- Lead optimization prioritization

In [1]:
def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False
IN_COLAB = in_colab()
print(f"{IN_COLAB=}")

IN_COLAB=False


## Installation Instructions

### Local Environment
```bash
conda create -n mmelon311 python=3.11 -y
conda activate mmelon311
git clone https://github.com/jmorrone/biomed-multi-view.git
pip install git+https://github.com/jmorrone/biomed-multi-view.git
pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.1.0 torchvision==0.16.0
pip install -f https://data.pyg.org/whl/torch-2.1.0+cu121.html "pyg_lib==0.4.0+pt21cu121" "torch_scatter==2.1.2+pt21cu121" "torch_cluster==1.6.3+pt21cu121" "torch_spline_conv==1.2.2+pt21cu121" --upgrade-strategy only-if-needed
pip install -q notebook ipykernel scikit-learn pandas openpyxl rdkit
```

In [2]:
if IN_COLAB:
    print("***** Select runtime 2025.07!!!")
    !git clone https://github.com/jmorrone/biomed-multi-view.git

In [3]:
if IN_COLAB:
    !pip install git+https://github.com/jmorrone/biomed-multi-view.git  

In [4]:
if IN_COLAB:
    import os
    os.chdir('biomed-multi-view')
    !pip install -r requirements.txt

## 1. Setup and configuration

Edit these paths and hyperparameters according to your needs.

In [5]:
import os
run_on_wdr91 = True

if IN_COLAB:
    MODELS_DIR = "/content/models/mmelon"
else:
    USER = os.getenv("USER")
    MODELS_DIR = f"/proj/ligand_ai/{USER}/models/mmelon"

if run_on_wdr91:
    NAME = "wdr91_asms"
    if IN_COLAB:
        DATA_PATH = "/content/data"
    else:
        DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1"
else:
    NAME = "pgk2_del_cdd"
    if IN_COLAB:
        DATA_PATH = "/content/data"
    else:
        DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/PGK2_DEL_cdd_to_creative"

MODEL_DIR = os.path.join(MODELS_DIR, NAME)

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model name

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
EPOCHS = 1 # 10  # Number of training epochs
BATCH_SIZE = 16  # Training batch size
LEARNING_RATE = 2e-5  # Learning rate
WEIGHT_DECAY = 0.01  # Weight decay for regularization
SEED = 42  # Random seed for reproducibility

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================
DEVICE = None  # None for auto-detection, or specify 'cuda' or 'cpu'

print("Configuration loaded successfully!")
print(f"Output model directory: {MODEL_DIR}")
print(f"Base Model: {BASE_MODEL}")
print(f"Training epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")

Configuration loaded successfully!
Output model directory: /proj/ligand_ai/ozery/models/mmelon/wdr91_asms
Base Model: ibm-research/biomed.sm.mv-te-84m
Training epochs: 1
Batch size: 16
Learning rate: 2e-05


## 2. Load and Prepare Data

Load the dataset splits and prepare them for training.

In [6]:
import pandas as pd
# Load split data

split_path = DATA_PATH

print(f"Loading split from: {split_path}")

train_df = pd.read_csv(f"{split_path}/train.csv")
val_df = pd.read_csv(f"{split_path}/val.csv")
test_df = pd.read_csv(f"{split_path}/test.csv")

print(f"\nSplit loaded successfully:")
print(f"  Train: {len(train_df):,} samples ({train_df['label'].mean()*100:.2f}% positive)")
print(f"  Val:   {len(val_df):,} samples ({val_df['label'].mean()*100:.2f}% positive)")
print(f"  Test:  {len(test_df):,} samples ({test_df['label'].mean()*100:.2f}% positive)")

Loading split from: /proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1

Split loaded successfully:
  Train: 305,332 samples (0.04% positive)
  Val:   33,926 samples (0.04% positive)
  Test:  234 samples (24.79% positive)


In [7]:
# Prepare data in MMELON format
# MMELON expects files named 'data_train.csv', 'data_val.csv', 'data_test.csv'

data_dir = split_path

# Save with MMELON-expected filenames if they don't exist
if not os.path.exists(f"{data_dir}/data_train.csv"):
    train_df.to_csv(f"{data_dir}/data_train.csv", index=False)
    print(f"Saved data_train.csv")

if not os.path.exists(f"{data_dir}/data_val.csv"):
    val_df.to_csv(f"{data_dir}/data_val.csv", index=False)
    print(f"Saved data_val.csv")

if not os.path.exists(f"{data_dir}/data_test.csv"):
    test_df.to_csv(f"{data_dir}/data_test.csv", index=False)
    print(f"Saved data_test.csv")

print(f"\nData directory: {data_dir}")


Data directory: /proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1


## 3. Prepare Data Loaders

In [8]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

from bmfm_sm.core.data_modules.namespace import LateFusionStrategy, TaskType
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel

/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-08 20:09:50,435 - rdkit - INFO - millie:22427627406016:0:0 - Enabling RDKit 2022.09.5 jupyter extensions


In [9]:
# Set random seed for reproducibility
pl.seed_everything(SEED)

# Create output directory
os.makedirs(MODEL_DIR, exist_ok=True)

print("="*80)
print("INITIALIZING MMELON MODEL")
print("="*80)

# Dataset arguments for MMELON framework
dataset_args = {
    'task_type': TaskType.CLASSIFICATION,
    'num_tasks': 1,  # Single task for binary classification
    'modalities': ['TEXT_MODEL', 'IMAGE_MODEL', 'GRAPH_2D_MODEL'],
    'smiles_col': 'smiles',
    'label_cols': ['label'],
    'split_col': None
}

# Create data modules
print("Creating data modules...")
train_dataset = MultiModalFinetuneDataPipeline(
    data_dir=str(data_dir),
    dataset_args=dataset_args,
    stage='train'
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_dataset.collate_fn,
    num_workers=4
)

val_dataset = MultiModalFinetuneDataPipeline(
    data_dir=str(data_dir),
    dataset_args=dataset_args,
    stage='val'
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=val_dataset.collate_fn,
    num_workers=4
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Seed set to 42
2026-03-08 20:09:51,245 - root - INFO - millie:22427627406016:0:0 - Working on task TaskType.CLASSIFICATION
2026-03-08 20:09:51,245 - root - INFO - millie:22427627406016:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


INITIALIZING MMELON MODEL
Creating data modules...


2026-03-08 20:09:55,421 - root - INFO - millie:22427627406016:0:0 - Working on task TaskType.CLASSIFICATION
2026-03-08 20:09:55,422 - root - INFO - millie:22427627406016:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


Train batches: 19084
Val batches: 2121


## 4. Initialize MMELON Model

In [10]:
FUSION_STRATEGY = LateFusionStrategy.ATTENTIONAL

# Model parameters
model_params = {
    'agg_arch': FUSION_STRATEGY.value[0],
    'agg_gate_input': FUSION_STRATEGY.value[1],
    'agg_weight_freeze': FUSION_STRATEGY.value[2],
    'inference_mode': False
}

# Finetuning arguments
finetuning_args = {
    'weight_freeze': 'unfrozen',
    'initialization': 'default',
    'head_arch': 'mlp',
    'use_norm': True,
    'head_dropout': 0.2
}

# Create Lightning module
print("Creating Lightning module...")
lightning_module = FineTuneLightningModule(
    base_model_class='bmfm_sm.predictive.modules.smmv_model.SmallMoleculeMultiView',
    model_params=model_params,
    task_type='classification',
    num_tasks=1,
    checkpoint_path=None,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    finetuning_args=finetuning_args
)

# Load pretrained weights from HuggingFace
print(f"Loading pretrained model from HuggingFace: {BASE_MODEL}")
pretrained_model = SmallMoleculeMultiViewModel.from_pretrained(
    FUSION_STRATEGY,
    model_path=BASE_MODEL,
    huggingface=True
)

lightning_module.model.load_state_dict(pretrained_model.state_dict(), strict=False)

print("Model initialized successfully!")

2026-03-08 20:09:56,177 - root - INFO - millie:22427627406016:0:0 - Initializing model with params {'agg_arch': 'coeff_mlp', 'agg_gate_input': 'coeff_mlp', 'agg_weight_freeze': None, 'inference_mode': False}


Creating Lightning module...


/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
2026-03-08 20:09:56,992 - root - INFO - millie:22427627406016:0:0 - Using coeff_mlp architecture for aggregator
2026-03-08 20:09:56,993 - root - INFO - millie:22427627406016:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-08 20:09:57,012 - root - INFO - millie:22427627406016:0:0 - Initialized the model with random weights as checkpoint_path was specified as None
2026-03-08 20:09:57,013 - root - INFO - mill

Loading pretrained model from HuggingFace: ibm-research/biomed.sm.mv-te-84m


2026-03-08 20:09:57,949 - root - INFO - millie:22427627406016:0:0 - Using coeff_mlp architecture for aggregator
2026-03-08 20:09:57,950 - root - INFO - millie:22427627406016:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-08 20:10:00,230 - root - INFO - millie:22427627406016:0:0 - in train False setting deterministic_eval = True


Model initialized successfully!


## 5. Configure Trainer

In [11]:
from pytorch_lightning.callbacks import ModelCheckpoint

# Setup callbacks
callbacks = [
    ModelCheckpoint(
        dirpath=MODEL_DIR,
        filename='mmelon-{epoch:02d}-{val_loss:.2f}',
        save_top_k=3,
        monitor='val_loss',
        mode='min',
        save_last=True,
        every_n_epochs=1
    )
]

# Create trainer
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=callbacks,
    default_root_dir=MODEL_DIR,
    accelerator='auto',
    devices=1,
    log_every_n_steps=10,
    val_check_interval=1.0,
    enable_checkpointing=True,
    enable_model_summary=True
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


## 6. Fine-Tune Model

In [12]:
%%time

print("="*80)
print("STARTING TRAINING")
print("="*80)

trainer.fit(lightning_module, train_loader, val_loader)

print("="*80)
print("TRAINING COMPLETE!")
print("="*80)

STARTING TRAINING


You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /gpfs/ZuRust/proj/ligand_ai/ozery/models/mmelon/wdr91_asms exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [1]

  | Name      | Type                    | Params | Mode  | FLOPs
----------------------------------------------------------------------
0 | model     | SmallMoleculeMultiView  | 84.1 M | train | 0    
1 | pred_head | MultiTaskPredictionHead | 461 K  | train | 0    
2 | criterion | BCEWithLogitsLoss       | 0      | train | 0    
----------------------------

Sanity Checking: |                                              | 0/? [00:00<?, ?it/s]

2026-03-08 20:10:03,113 - root - INFO - millie:22427627406016:0:0 - in train False setting deterministic_eval = True


Sanity Checking DataLoader 0:  50%|██████████▌          | 1/2 [00:01<00:01,  0.75it/s]

...

Epoch 0: 100%|████| 19084/19084 [32:42<00:00,  9.72it/s, v_num=1, train_loss=0.000495]

2026-03-08 20:42:48,357 - root - INFO - millie:22427627406016:0:0 - in train False setting deterministic_eval = True


Epoch 0: 100%|█| 19084/19084 [34:05<00:00,  9.33it/s, v_num=1, train_loss=0.000495, va

/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 6. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|█| 19084/19084 [34:09<00:00,  9.31it/s, v_num=1, train_loss=0.000495, va
TRAINING COMPLETE!
CPU times: user 32min 23s, sys: 1min 32s, total: 33min 55s
Wall time: 34min 15s


In [ ]:
os.listdir(MODEL_DIR)